EXPERIMENT:
SEQ_LEN = 96
PRED_LEN = 48

In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
data_path = "..\\01_dataset\\ETTh1.csv"

df = pd.read_csv(data_path)

df.head()

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


In [3]:
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

df.head()

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


In [4]:
train_size = int(len(df) * 0.7)
val_size = int(len(df) * 0.2)

train_df = df[:train_size]
val_df = df[train_size:train_size + val_size]
test_df = df[train_size + val_size:]

In [5]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

In [7]:
joblib.dump(scaler, "..\\01_dataset\\preprocessed-1\\scaler.save")

['..\\01_dataset\\preprocessed-1\\scaler.save']

SLIDING WINDOW

In [8]:
def create_sequences(data, seq_len=96, pred_len=48, target_index=-1):
    X = []
    y = []

    for i in range(len(data) - seq_len - pred_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len:i+seq_len+pred_len, target_index])

    return np.array(X), np.array(y)

SEQUENCE

In [9]:
SEQ_LEN = 96
PRED_LEN = 48

X_train, y_train = create_sequences(train_scaled, SEQ_LEN, PRED_LEN)
X_val, y_val = create_sequences(val_scaled, SEQ_LEN, PRED_LEN)
X_test, y_test = create_sequences(test_scaled, SEQ_LEN, PRED_LEN)

print(X_train.shape, y_train.shape)

(12050, 96, 7) (12050, 48)


In [10]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [11]:
torch.save(X_train, "..\\01_dataset\\preprocessed-1\\X_train.pt")
torch.save(y_train, "..\\01_dataset\\preprocessed-1\\y_train.pt")

torch.save(X_val, "..\\01_dataset\\preprocessed-1\\X_val.pt")
torch.save(y_val, "..\\01_dataset\\preprocessed-1\\y_val.pt")

torch.save(X_test, "..\\01_dataset\\preprocessed-1\\X_test.pt")
torch.save(y_test, "..\\01_dataset\\preprocessed-1\\y_test.pt")